# OWCS Result Screen Extraction Pipeline

This notebook contains the AI-assisted data extraction and validation pipeline used for the OWCS Korea Stat Lab.

The pipeline converts broadcast result-screen screenshots into structured player-level statistics using the following process:

**Result Screens → AI-Assisted Extraction → Deterministic Validation → Structured Dataset → Player Aggregation → Per-10 Statistics**

AI-generated output is treated as an intermediate extraction result rather than ground truth. Extracted records must pass deterministic validation checks before entering the processed dataset.

### Pipeline Overview

1. Load a canonical roster reference
2. Extract scoreboard data from each result screen
3. Parse the model response into a fixed schema
4. Validate player identities, numeric fields, playtime, and scoreboard consistency
5. Exclude failed extractions
6. Preserve source-file traceability
7. Aggregate validated records by player
8. Calculate accumulated and Per-10-minute statistics

In [ ]:
import os
import re
import csv
import time
from pathlib import Path
from datetime import datetime

import pandas as pd
import google.generativeai as genai


# ============================================================
# API CONFIGURATION
# ============================================================

# Set GEMINI_API_KEY in your environment before running.
#
# Example:
#   export GEMINI_API_KEY="your_api_key_here"
#
# Never hardcode API keys directly in this notebook.

API_KEY = os.getenv("GEMINI_API_KEY")

if not API_KEY:
    raise RuntimeError(
        "GEMINI_API_KEY is not set. "
        "Set it as an environment variable before running this notebook."
    )

MODEL_NAME = os.getenv("GEMINI_MODEL", "gemini-3.5-flash")

genai.configure(api_key=API_KEY)
model = genai.GenerativeModel(MODEL_NAME)


# ============================================================
# INPUT / OUTPUT PATHS
# ============================================================

# Replace these example paths with your local file locations.

# Folder containing result-screen screenshots.
BASE_DIR = Path.cwd() if (Path.cwd() / "result_screen_s3").exists() else Path.cwd().parent
# Folder containing result-screen screenshots.
IMAGE_DIR = BASE_DIR / "result_screen_s3"
# Workbook containing the canonical TEAM INFO sheet.
ROSTER_XLSX = BASE_DIR / "OWCS_STAT_LAB_26_KR_STAGE3.xlsx"
# Pipeline outputs.
OUTPUT_DIR = BASE_DIR / "output"
PARSED_XLSX = OUTPUT_DIR / "validated_results.xlsx"
CLEAN_CSV = OUTPUT_DIR / "clean_raw_data.csv"
REVIEW_CSV = OUTPUT_DIR / "validation_report.csv"
CHECKPOINT_CSV = OUTPUT_DIR / "checkpoint.csv"
MASTER_XLSX = OUTPUT_DIR / "player_statistics.xlsx"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


# ============================================================
# PIPELINE SETTINGS
# ============================================================

MAX_RETRIES = 3
REQUEST_DELAY_SEC = 1

CHUNK_SIZE = 6
CHUNK_COOLDOWN_SEC = 15

MIN_PER10_MINUTES = 30

## Extraction Schema

Each result screen is expected to contain exactly 10 player records.

The extraction model returns the following fields:

| Field | Description |
|---|---|
| Team | Team abbreviation shown on screen |
| Player | Player ID shown on screen |
| Position | Role shown for the player on that map |
| Map | Map name |
| Map Type | Game mode |
| Elim | Eliminations |
| Death | Deaths |
| Assists | Assists |
| Damage | Damage dealt |
| Heal | Healing |
| Mitigated | Damage mitigated |
| Playtime | Map playtime |

Player identity and team membership are validated against a canonical roster after extraction.

In [ ]:
COLUMNS = [
    "Team",
    "Player",
    "Position",
    "Map",
    "Map Type",
    "Elim",
    "Death",
    "Assists",
    "Damage",
    "Heal",
    "Mitigated",
    "Playtime",
]

NUMERIC_COLS = [
    "Elim",
    "Death",
    "Assists",
    "Damage",
    "Heal",
    "Mitigated",
]

VALID_POSITIONS = {
    "TANK",
    "DPS",
    "SPT",
}

VALID_MAP_TYPES = {
    "Control",
    "Hybrid",
    "Escort",
    "Flashpoint",
    "Push",
    "Clash",
}


PROMPT = """
This image is an Overwatch esports result/scoreboard screen.

Extract exactly the 10 player rows and return CSV only.
Do not include a header, markdown, or explanation.

Columns, in this exact order:
Team,Player,Position,Map,Map Type,Elim,Death,Assists,Damage,Heal,Mitigated,Playtime

Rules:
- Return exactly 10 rows, one per player.
- Team: use the team abbreviation shown on screen.
- Player: copy the player ID exactly as displayed. Do not guess or autocorrect.
- Position: TANK, DPS, or SPT only.
- Map: use the map name shown on screen.
- Map Type: Control, Hybrid, Escort, Flashpoint, Push, or Clash.
- Carefully map scoreboard E/A/D values to Elim/Assists/Death.
- Damage, Heal, and Mitigated must contain digits only.
- Do not include thousands separators.
- Playtime must use HH:MM:SS format.
- All 10 players should have the same map and playtime.
- If a value cannot be read confidently, leave that field empty.
- Never invent a value.
""".strip()

## Canonical Roster Reference

The `TEAM INFO` sheet is used as the canonical reference for player identity and team membership.

The roster reference is also used to store each player's primary roster role and detailed analytical role.

Map-level `Position` is preserved separately because a player may compete in a different role during an individual map.

In [ ]:
def load_roster(path):
    """
    Load canonical player information from the TEAM INFO sheet.

    Expected columns:
    Team Name | Shortened | Player Name | Player Position | Detailed
    """

    if not path.exists():
        print(
            f"⚠️ Roster file not found: {path}. "
            "Player/team validation will be limited."
        )
        return {}

    team_info = pd.read_excel(
        path,
        sheet_name="TEAM INFO"
    )

    roster = {}

    current_team = None
    current_short = None

    for _, row in team_info.iterrows():

        values = row.tolist()

        team = values[0] if len(values) > 0 else None
        short = values[1] if len(values) > 1 else None
        player = values[2] if len(values) > 2 else None
        position = values[3] if len(values) > 3 else None
        detailed = values[4] if len(values) > 4 else None

        if pd.notna(team) and pd.notna(short):
            current_team = str(team).strip()
            current_short = str(short).strip()

        if pd.notna(player):

            player_id = str(player).strip()

            roster[player_id] = {
                "Team": current_short,
                "Roster Position": (
                    str(position).strip()
                    if pd.notna(position)
                    else None
                ),
                "Detailed Position": (
                    str(detailed).strip()
                    if pd.notna(detailed)
                    else None
                ),
            }

    return roster


ROSTER = load_roster(ROSTER_XLSX)

print(
    f"Loaded {len(ROSTER)} players "
    "from TEAM INFO."
)

In [ ]:
def clean_model_text(text):
    """Remove optional Markdown code fences."""

    text = (text or "").strip()

    text = re.sub(
        r"^```(?:csv)?\s*",
        "",
        text,
        flags=re.I,
    )

    text = re.sub(
        r"\s*```$",
        "",
        text,
    )

    return text.strip()


def parse_csv_response(text):
    """
    Parse the model response into the expected schema.

    Lines with an unexpected number of fields are retained
    separately as malformed lines for validation.
    """

    rows = []
    bad_lines = []

    reader = csv.reader(
        clean_model_text(text).splitlines()
    )

    for line_no, items in enumerate(
        reader,
        start=1,
    ):

        items = [
            item.strip()
            for item in items
        ]

        if not any(items):
            continue

        if len(items) != len(COLUMNS):

            bad_lines.append(
                (line_no, items)
            )

            continue

        rows.append(
            dict(zip(COLUMNS, items))
        )

    return rows, bad_lines


def parse_number(value):
    """
    Convert an extracted numeric field to int or float.

    Invalid values return None instead of being silently
    converted to zero.
    """

    s = str(value).strip().replace(",", "")

    if not s:
        return None

    if not re.fullmatch(
        r"-?\d+(?:\.\d+)?",
        s,
    ):
        return None

    return (
        float(s)
        if "." in s
        else int(s)
    )


def normalize_time(value):
    """Normalize playtime to HH:MM:SS."""

    s = str(value).strip()

    if not s:
        return None

    parts = s.split(":")

    try:

        if len(parts) == 2:
            h = 0
            m, sec = map(int, parts)

        elif len(parts) == 3:
            h, m, sec = map(int, parts)

        else:
            return None

        if (
            min(h, m, sec) < 0
            or m >= 60
            or sec >= 60
        ):
            return None

        return f"{h:02d}:{m:02d}:{sec:02d}"

    except ValueError:
        return None

## Deterministic Validation

AI output is not treated as ground truth.

Each extracted result screen is checked for:

- malformed CSV rows
- exactly 10 player records
- duplicate player IDs
- unknown player IDs
- invalid numeric fields
- invalid playtime
- invalid or unusual categorical values
- inconsistent map information
- inconsistent playtime across the result screen
- team membership against the canonical roster

Validation produces one of three statuses:

- **PASS** — no detected issues
- **WARNING** — usable record with a detectable non-fatal inconsistency
- **FAIL** — record excluded from the clean dataset

Both PASS and WARNING records remain traceable to their original source screenshot.

In [ ]:
import sys

# Canonical map names (pipeline/names.py): the model copies the on-screen spelling
# ('ROUTE 66', 'ESPERANÇA'), so maps are folded to one name before any comparison.
sys.path.insert(0, str(BASE_DIR / "pipeline"))
from names import MAP_TYPES, canonical_map, match_player  # noqa: E402

# Explicit aliases are only for misreads that are not look-alike characters (SORE/SOAE).
# Look-alike spellings (0/O, 1/I/L, 5/S, 8/B) are matched to TEAM INFO automatically below.
# SORE is PF's support SOAE (SPT row; mapping it to SORI made SORI appear twice). Leo, 2026-10-14.
PLAYER_ALIASES = {
    "CHORONG": "CH0R0NG",
    "SORE": "SOAE",
    "TREJONG": "TAEJONG",
}

TEAM_ALIASES = {
    "02": "O2",
    "RT": "SEJ",
}

def validate_image_rows(
    rows,
    bad_lines,
    filename,
):
    """
    Validate and normalize one extracted scoreboard.

    FAIL records are excluded from the clean dataset.
    WARNING records remain available for review.
    """

    issues = []
    corrections = []
    fatal = False

    # --------------------------------------------------------
    # Structural validation
    # --------------------------------------------------------

    if bad_lines:
        issues.append(f"{len(bad_lines)} malformed CSV line(s)")
        fatal = True

    if len(rows) != 10:
        issues.append(f"expected 10 rows, got {len(rows)}")
        fatal = True

    # --------------------------------------------------------
    # Apply aliases to rows before validation
    # --------------------------------------------------------
    for r in rows:
        raw_p = str(r.get("Player", "")).strip()
        raw_t = str(r.get("Team", "")).strip()
        if raw_p.upper() in PLAYER_ALIASES:
            corr_p = PLAYER_ALIASES[raw_p.upper()]
            corrections.append(f"Player alias: {raw_p}→{corr_p}")
            r["Player"] = corr_p
        if raw_t.upper() in TEAM_ALIASES:
            corr_t = TEAM_ALIASES[raw_t.upper()]
            corrections.append(f"Team alias: {raw_t}→{corr_t}")
            r["Team"] = corr_t

    # --------------------------------------------------------
    # Auto-match IDs that differ from TEAM INFO only by look-alike characters
    # (e.g. ALIEN -> A1IEN). Only a unique match is applied; anything else stays
    # as extracted and fails below as an unknown player. Every change is logged.
    # --------------------------------------------------------
    if ROSTER:
        for r in rows:
            raw_p = str(r.get("Player", "")).strip()
            if raw_p and raw_p not in ROSTER:
                matched = match_player(raw_p, list(ROSTER))
                if matched:
                    corrections.append(f"Player auto-matched to TEAM INFO: {raw_p}→{matched}")
                    r["Player"] = matched

    # --------------------------------------------------------
    # Player validation
    # --------------------------------------------------------

    players = [
        str(row.get("Player", "")).strip()
        for row in rows
    ]

    missing_players = [
        idx for idx, player in enumerate(players, start=1)
        if not player
    ]
    if missing_players:
        issues.append("missing player ID at row(s): " + ", ".join(map(str, missing_players)))
        fatal = True

    dupes = sorted({
        player for player in players
        if player and players.count(player) > 1
    })
    if dupes:
        issues.append("duplicate player(s): " + ", ".join(dupes))
        fatal = True

    unknown = sorted({
        player for player in players
        if player and ROSTER and player not in ROSTER
    })
    if unknown:
        issues.append("unknown player(s): " + ", ".join(unknown))
        fatal = True

    # --------------------------------------------------------
    # Row-level validation
    # --------------------------------------------------------

    normalized = []
    map_renames = set()

    for idx, row in enumerate(rows, start=1):
        record = dict(row)
        player = str(record.get("Player", "")).strip()
        record["Player"] = player
        row_issues = []

        # Numeric fields
        for col in NUMERIC_COLS:
            value = parse_number(record.get(col))
            if value is None:
                row_issues.append(f"{col} invalid")
                fatal = True
            record[col] = value

        # Playtime
        playtime = normalize_time(record.get("Playtime"))
        if playtime is None:
            row_issues.append("Playtime invalid")
            fatal = True
        record["Playtime"] = playtime

        # Position
        pos = str(record.get("Position", "")).strip()
        record["Position"] = pos
        if pos not in VALID_POSITIONS:
            row_issues.append(f"Position invalid: {pos}")

        # Map: fold case / accents / punctuation to the canonical name.
        # An unknown map is kept as extracted and reported, never guessed.
        raw_map = str(record.get("Map", "")).strip()
        map_name = canonical_map(raw_map)
        if map_name is None:
            row_issues.append(f"Map unknown: {raw_map}")
            record["Map"] = raw_map
        else:
            if map_name != raw_map:
                map_renames.add(f"{raw_map}→{map_name}")
            record["Map"] = map_name

        # Map Type
        map_type = str(record.get("Map Type", "")).strip()
        record["Map Type"] = map_type
        if map_type not in VALID_MAP_TYPES:
            row_issues.append(f"Map Type unusual: {map_type}")
        elif map_name and MAP_TYPES[map_name] != map_type:
            row_issues.append(f"Map Type {map_type} != {MAP_TYPES[map_name]} for {map_name}")

        # Canonical roster cross-reference
        if player in ROSTER:
            canonical = ROSTER[player]
            canonical_team = canonical.get("Team")
            roster_pos = canonical.get("Roster Position")
            detailed_pos = canonical.get("Detailed Position")

            extracted_team = str(record.get("Team", "")).strip()
            if canonical_team and extracted_team != canonical_team:
                corrections.append(f"row {idx} ({player}): Team corrected {extracted_team}→{canonical_team}")
            record["Team"] = canonical_team or extracted_team

            if roster_pos and pos and pos != roster_pos:
                corrections.append(f"row {idx} ({player}): Map role differs ({pos} vs {roster_pos})")
            record["Roster Position"] = roster_pos
            record["Detailed Position"] = detailed_pos
        else:
            record["Roster Position"] = None
            record["Detailed Position"] = None

        if row_issues:
            issues.append(f"row {idx} ({player or 'UNKNOWN'}): " + "; ".join(row_issues))

        normalized.append(record)

    if map_renames:
        corrections.append("Map name: " + ", ".join(sorted(map_renames)))

    # --------------------------------------------------------
    # Scoreboard-level consistency (maps compared after canonicalization)
    # --------------------------------------------------------
    for col in ["Map", "Map Type", "Playtime"]:
        values = {
            str(record.get(col))
            for record in normalized
            if record.get(col) not in (None, "")
        }
        if len(values) > 1:
            issues.append(f"inconsistent {col}: {sorted(values)}")
            fatal = True

    if fatal:
        status = "FAIL"
    elif issues:
        status = "WARNING"
    else:
        status = "PASS"

    if corrections:
        print(f"[INFO] {filename}: " + " | ".join(corrections))

    return normalized, status, issues


In [ ]:
def extract_one_image(path):
    """
    Extract and validate one result-screen screenshot.
    """

    last_error = None

    for attempt in range(
        1,
        MAX_RETRIES + 1,
    ):

        try:

            uploaded = genai.upload_file(
                path=str(path)
            )

            response = model.generate_content(
                [uploaded, PROMPT]
            )

            rows, bad_lines = (
                parse_csv_response(
                    response.text
                )
            )

            normalized, status, issues = (
                validate_image_rows(
                    rows,
                    bad_lines,
                    path.name,
                )
            )

            return (
                normalized,
                status,
                issues,
                response.text,
            )


        except Exception as error:

            last_error = error

            if attempt < MAX_RETRIES:

                wait = 2 ** (
                    attempt - 1
                )

                print(
                    f"    retry "
                    f"{attempt}/{MAX_RETRIES} "
                    f"after error: "
                    f"{error} ({wait}s)"
                )

                time.sleep(wait)


    return (
        [],
        "FAIL",
        [
            "API/parsing error after "
            f"{MAX_RETRIES} attempts: "
            f"{last_error}"
        ],
        "",
    )

In [ ]:
clean_rows = []
failed_rows = []
report = []
processed_files = set()

FAILED_CSV = OUTPUT_DIR / "failed_raw_data.csv"

# ============================================================
# RESUME FROM CHECKPOINT
# ============================================================

if CHECKPOINT_CSV.exists():
    checkpoint = pd.read_csv(CHECKPOINT_CSV)
    if "Source File" in checkpoint.columns:
        processed_files = set(
            checkpoint["Source File"].dropna().astype(str)
        )
        clean_rows = checkpoint.to_dict("records")
        print(
            "Resuming from checkpoint: "
            f"{len(processed_files)} image(s) already accepted."
        )

# ============================================================
# DISCOVER SCREENSHOTS
# ============================================================

if not IMAGE_DIR.exists():
    raise FileNotFoundError(f"Image folder not found: {IMAGE_DIR}")

image_files = sorted([
    path for path in IMAGE_DIR.iterdir()
    if path.suffix.lower() in {".png", ".jpg", ".jpeg"}
])

print(f"📸 {len(image_files)} screenshots found.")

pending = [
    path for path in image_files
    if path.name not in processed_files
]

print(f"⏳ {len(pending)} screenshots pending processing.")

# ============================================================
# RUN EXTRACTION
# ============================================================

for idx, path in enumerate(pending, start=1):
    print(f"[{idx}/{len(pending)}] {path.name}")

    rows, status, issues, raw_response = extract_one_image(path)

    report.append({
        "Source File": path.name,
        "Status": status,
        "Rows Parsed": len(rows),
        "Issues": " | ".join(issues),
        "Checked At": datetime.now().isoformat(timespec="seconds"),
    })

    if status != "FAIL":
        for row in rows:
            row["Source File"] = path.name
            row["Validation Status"] = status
            clean_rows.append(row)

        pd.DataFrame(clean_rows).to_csv(
            CHECKPOINT_CSV,
            index=False,
            encoding="utf-8-sig",
        )
        print(f"    {status}: accepted {len(rows)} rows")
    else:
        print("    FAIL: excluded from clean dataset (saved to failed_raw_data)")
        for row in rows:
            row["Source File"] = path.name
            row["Validation Status"] = status
            row["Issues"] = " | ".join(issues)
            failed_rows.append(row)
        for issue in issues:
            print("      -", issue)

    time.sleep(REQUEST_DELAY_SEC)

    if idx % CHUNK_SIZE == 0 and idx < len(pending):
        print(f"⏳ chunk cooldown: {CHUNK_COOLDOWN_SEC}s")
        time.sleep(CHUNK_COOLDOWN_SEC)



In [ ]:
clean_df = pd.DataFrame(clean_rows)
failed_df = pd.DataFrame(failed_rows)
report_df = pd.DataFrame(report)

if not clean_df.empty:
    clean_df.to_csv(CLEAN_CSV, index=False, encoding="utf-8-sig")

if not failed_df.empty:
    failed_df.to_csv(FAILED_CSV, index=False, encoding="utf-8-sig")

if not report_df.empty:
    report_df.to_csv(REVIEW_CSV, index=False, encoding="utf-8-sig")

with pd.ExcelWriter(PARSED_XLSX) as writer:
    if not clean_df.empty:
        clean_df.to_excel(writer, sheet_name="Clean Raw Data", index=False)
    if not failed_df.empty:
        failed_df.to_excel(writer, sheet_name="Failed Raw Data", index=False)
    if not report_df.empty:
        report_df.to_excel(writer, sheet_name="Validation Report", index=False)

print(f"✅ Clean rows: {len(clean_df)}")
if not failed_df.empty:
    print(f"⚠️ Failed rows backed up: {len(failed_df)} (in {FAILED_CSV})")
print(f"📋 Validation report: {REVIEW_CSV}")
print(f"📗 Validated workbook: {PARSED_XLSX}")



# Statistical Processing

The following step generates accumulated and Per-10-minute player statistics from the validated map-level dataset.

Both accumulated and Per-10 statistics are derived from the same clean source to maintain consistency.

Player ID is used as the primary aggregation key.

Map-level `Position` is intentionally not treated as part of player identity because some players may compete in multiple roles during the competition.

In [ ]:
# ============================================================
# LOAD VALIDATED DATA
# ============================================================

raw = pd.read_excel(
    PARSED_XLSX,
    sheet_name="Clean Raw Data",
)


numeric_cols = [
    "Elim",
    "Death",
    "Assists",
    "Damage",
    "Heal",
    "Mitigated",
]


for col in numeric_cols:

    raw[col] = pd.to_numeric(
        raw[col],
        errors="raise",
    )


# ============================================================
# PLAYTIME
# ============================================================

def time_to_seconds(value):
    """Convert HH:MM:SS or MM:SS to seconds."""

    if pd.isna(value):

        raise ValueError(
            "Missing Playtime in "
            "accepted clean data."
        )


    parts = str(value).strip().split(":")


    if len(parts) == 3:

        h, m, sec = map(
            int,
            parts,
        )


    elif len(parts) == 2:

        h = 0

        m, sec = map(
            int,
            parts,
        )


    else:

        raise ValueError(
            f"Invalid Playtime: {value}"
        )


    if (
        min(h, m, sec) < 0
        or m >= 60
        or sec >= 60
    ):

        raise ValueError(
            f"Invalid Playtime: {value}"
        )


    return (
        h * 3600
        + m * 60
        + sec
    )


raw["Playtime_Seconds"] = (
    raw["Playtime"]
    .apply(time_to_seconds)
)


# ============================================================
# PLAYER IDENTITY
# ============================================================

# Player ID is the aggregation key.
#
# Map-level Position is intentionally excluded because
# players may compete in multiple roles.

identity_cols = ["Team"]


if "Roster Position" in raw.columns:
    identity_cols.append(
        "Roster Position"
    )


if "Detailed Position" in raw.columns:
    identity_cols.append(
        "Detailed Position"
    )


identity_check = (
    raw.groupby("Player")[
        identity_cols
    ]
    .nunique(dropna=False)
)


problem_players = (
    identity_check[
        (identity_check > 1)
        .any(axis=1)
    ]
)


if not problem_players.empty:

    raise ValueError(
        "Canonical identity conflict "
        "for player(s): "
        + ", ".join(
            problem_players
            .index
            .astype(str)
        )
    )


identity = (
    raw.groupby(
        "Player",
        as_index=False,
    )[identity_cols]
    .first()
)


# ============================================================
# ACCUMULATED STATISTICS
# ============================================================

stats = (
    raw.groupby(
        "Player",
        as_index=False,
    )[
        numeric_cols
        + ["Playtime_Seconds"]
    ]
    .sum()
)


total = identity.merge(
    stats,
    on="Player",
    how="inner",
)


total["Playtime_Min"] = (
    total["Playtime_Seconds"]
    / 60
)


front = [
    "Team",
    "Player",
]


for col in [
    "Roster Position",
    "Detailed Position",
]:

    if col in total.columns:
        front.append(col)


total = total[
    front
    + numeric_cols
    + [
        "Playtime_Seconds",
        "Playtime_Min",
    ]
]


# ============================================================
# PER-10-MINUTE STATISTICS
# ============================================================

per10 = total[
    front
    + [
        "Playtime_Seconds",
        "Playtime_Min",
    ]
].copy()


for col in numeric_cols:

    per10[f"{col} / 10"] = (
        total[col]
        / total["Playtime_Seconds"]
        * 600
    ).round(2)


# Keep all calculated values in the processed dataset.
#
# The eligibility flag is used by rankings and
# visualizations to exclude small samples.

per10["Eligible_30min"] = (
    per10["Playtime_Min"]
    >= MIN_PER10_MINUTES
)


# ============================================================
# EXPORT
# ============================================================

with pd.ExcelWriter(
    MASTER_XLSX
) as writer:

    total.to_excel(
        writer,
        sheet_name="TOTAL STATISTICS",
        index=False,
    )

    per10.to_excel(
        writer,
        sheet_name="Per10 STATISTICS",
        index=False,
    )

    raw.to_excel(
        writer,
        sheet_name="Clean Raw Data",
        index=False,
    )


print(
    f"✅ {len(total)} players aggregated "
    "from the same validated raw dataset."
)

print(
    "✅ Per10 eligibility threshold: "
    f"{MIN_PER10_MINUTES} minutes."
)

print(
    f"📗 Master workbook: "
    f"{MASTER_XLSX}"
)